In [12]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [13]:
import os
os.chdir("/content/gdrive/My Drive/TL_A/DeepKale/experiments_0.2/")

In [14]:
%%capture
!pip install optuna torch plotly botorch

In [15]:
# Import Packages
import os
import pandas as pd
import datetime
import time
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import optuna
import matplotlib.pyplot as plt
import pickle as pkl

In [16]:
from src.preprocessing.data_preparation import DataPreparation
from src.preprocessing.feature_scaling import robust_scaling, min_max_scaling
from src.preprocessing.feature_engineering import feature_encoding, create_temporal_features, create_lag_features, expanding_mean_std_weighted_avg
from src.utils.helper_functions import get_approach

In [17]:
warnings.filterwarnings("ignore")

# Set seed for numpy
SEED = 1

# # Set seed for PyTorch
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

Device: cpu



In [19]:
if os.path.exists('/mnt/work/dkale/dkale_Colab/experiments_0.2/'):
    # Define base directory path
    root_dir = '/mnt/work/dkale/dkale_Colab/experiments_0.2/'
else:
    root_dir = '/content/gdrive/My Drive/TL_A/DeepKale/experiments_0.2/'

current_date = datetime.datetime.now().strftime("%d%m%Y")
approach = get_approach(model_type=0)

In [20]:
N_FUTURE = 1
N_PAST = 7

In [21]:
# Define subdirectories
root_data_dir = root_dir + 'data/'
preprocessed_dir = root_data_dir + 'preprocessed/ACN/'

In [22]:
base_df = pd.read_csv(rf'{preprocessed_dir}acn_caltech_jpl_0.2_0.1.22_06_2023.csv')

In [23]:
base_df = feature_encoding(base_df)

In [24]:
base_df['siteID_19'] = 0

In [25]:
base_df = create_temporal_features(base_df)

In [26]:
base_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16548 entries, 0 to 16547
Data columns (total 30 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   connectionTime                16548 non-null  datetime64[ns]
 1   kWhDelivered_Smoothed         16548 non-null  float64       
 2   total_charging_time_Smoothed  16548 non-null  float64       
 3   idle_time_Smoothed            16548 non-null  float64       
 4   siteID_0                      16548 non-null  int64         
 5   siteID_1                      16548 non-null  int64         
 6   siteID_1_2                    16548 non-null  int64         
 7   siteID_2                      16548 non-null  int64         
 8   siteID_19                     16548 non-null  int64         
 9   Hour_of_Day                   16548 non-null  int64         
 10  Day_Of_Week                   16548 non-null  int64         
 11  Day_Of_year                 

In [27]:
base_df = create_lag_features(dataframe = base_df, target = base_df['kWhDelivered_Smoothed'], thres=0.15)

In [28]:
base_df = expanding_mean_std_weighted_avg(dataframe = base_df, window_size = 2)

In [34]:
help(min_max_scaling)

Help on function min_max_scaling in module src.preprocessing.feature_scaling:

min_max_scaling(df, minmax_cols=[], sin_cos_cols=[])



In [35]:
base_robust_cols = ['kWhDelivered_Smoothed', 'total_charging_time_Smoothed', 'idle_time_Smoothed', 'lag_1', 'lag_2', 'expanding_mean', 'expanding_std', 'weighted_avg']
base_sin_cos_cols = ['Hour_of_Day', 'Day_Of_Week', 'Day_Of_year', 'Month_Of_Year']
base_scaled_df, base_minmin_max_scaling_params =min_max_scaling(df = base_df, minmax_cols = base_robust_cols, sin_cos_cols = base_sin_cos_cols)

In [36]:
base_scaled_df

,connectionTime,kWhDelivered_Smoothed,total_charging_time_Smoothed,idle_time_Smoothed,siteID_0,siteID_1,siteID_1_2,siteID_2,siteID_19,Time_of_day_0_4,...,expanding_std,weighted_avg,Hour_of_Day_sin,Hour_of_Day_cos,Day_Of_Week_sin,Day_Of_Week_cos,Day_Of_year_sin,Day_Of_year_cos,Month_Of_Year_sin,Month_Of_Year_cos
2,2018-04-25 08:00:00,0.287116,0.266520,0.520982,0,0,0,1,0,0,...,0.000000,0.341536,0.816970,-0.576680,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
3,2018-04-25 09:00:00,0.155045,0.269232,0.567425,0,0,0,1,0,0,...,0.265549,0.323725,0.631088,-0.775711,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
4,2018-04-25 10:00:00,0.296392,0.465599,0.351162,0,0,0,1,0,0,...,0.273233,0.269216,0.398401,-0.917211,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
5,2018-04-25 11:00:00,0.160130,0.260116,0.725258,0,0,0,1,0,0,...,0.317028,0.334217,0.136167,-0.990686,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
6,2018-04-25 12:00:00,0.075012,0.094847,0.000207,0,0,0,1,0,0,...,0.502527,0.175460,-0.136167,-0.990686,0.866025,-0.5,0.917584,-0.397543,0.866025,-5.000000e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16543,2020-03-14 13:00:00,0.042540,0.024696,0.003690,1,0,0,0,0,0,...,0.999644,0.094418,-0.398401,-0.917211,-0.866025,0.5,0.956235,0.292600,1.000000,6.123234e-17
16544,2020-03-14 14:00:00,0.021270,0.012348,0.001845,0,1,0,0,0,0,...,0.999639,0.047209,-0.631088,-0.775711,-0.866025,0.5,0.956235,0.292600,1.000000,6.123234e-17
16545,2020-03-14 15:00:00,0.208650,0.118653,0.000198,0,0,0,1,0,0,...,0.999609,0.111503,-0.816970,-0.576680,-0.866025,0.5,0.956235,0.292600,1.000000,6.123234e-17
16546,2020-03-14 16:00:00,0.105815,0.130636,0.031668,0,0,0,1,0,0,...,0.999575,0.232210,-0.942261,-0.334880,-0.866025,0.5,0.956235,0.292600,1.000000,6.123234e-17


In [37]:
base_minmin_max_scaling_params

{'kWhDelivered_Smoothed': (3.825163482815226e-12, 31.661760295811032)}